Step 1: 

```bash
CUDA_VISIBLE_DEVICES=0 vllm serve Qwen/Qwen2.5-7B-Instruct \
    --host 0.0.0.0 \
    --port 8081 \
    --gpu-memory-utilization 0.85 \
    --enable-prefix-caching \
    --dtype bfloat16 \
    --max_model_len 8000
```

```bash
CUDA_VISIBLE_DEVICES=1 vllm serve Skywork/Skywork-o1-Open-PRM-Qwen-2.5-7B \
    --host 0.0.0.0 \
    --port 8082 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

```bash
CUDA_VISIBLE_DEVICES=2 vllm serve Qwen/Qwen2.5-Math-PRM-7B \
    --host 0.0.0.0 \
    --port 8083 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = "EMPTY"
OPENAI_API_BASE = "http://localhost:{PORT}/v1"

causal_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8081),
)
causal_model = causal_client.models.list().data[0].id

skywork_prm_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8082),
)
skywork_prm_model = skywork_prm_client.models.list().data[0].id

qwen_prm_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8083),
)
qwen_prm_model = qwen_prm_client.models.list().data[0].id


In [ ]:
import re
import pandas as pd
from tqdm.auto import tqdm
from functools import partial
from transformers import AutoTokenizer
from multiprocessing import Pool, cpu_count
from constants.prompts_constants import (
    VERBOSE_TASK, CONSISE_TASK, EQ_TO_TEXT_TASK, CHANGE_NUMBERS_TASK
)

from utils.prompt_utils import get_augmentation_prompt, get_equivalence_prompt
from utils.io_utils import prepare_input, derive_step_rewards_vllm, prepare_batch_input_for_model

In [ ]:
df = pd.read_parquet("data/processbench.parquet")

df.keys()

Index(['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label',
       'split', 'steps_len', 'per_step_len', 'Qwen2.5-Math-PRM-7B',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B'],
      dtype='object')

In [ ]:
def augmentor(index_row, task_text, client, model):
    _, row = index_row

    question, steps = row["problem"], row["steps"]
    prompt          = get_augmentation_prompt(question, steps, task_text)

    # call OpenAI
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],

    )
    content = resp.choices[0].message.content

    # grab the <response>…</response> block
    m = re.search(r"<response>(.*?)</response>", content, re.DOTALL)
    if not m:
        return {"aug_question":"", "aug_steps":[]}

    body = m.group(1).strip()

    # extract question
    q_m = re.search(r"<question>(.*?)</question>", body, re.DOTALL)
    aug_question = q_m.group(1).strip() if q_m else ""

    # extract steps: find all <step#>…</step#>
    aug_steps = re.findall(r"<step\d+>(.*?)</step\d+>", body, re.DOTALL)
    aug_steps = [s.strip() for s in aug_steps]

    return {"aug_question": aug_question, "aug_steps": aug_steps}

def equivalence_check(pair, client, model):
    (_, rowA), (_, rowB) = pair
    qA, stepsA = rowA["problem"], rowA["steps"]
    qB, stepsB = rowB["problem"], rowB["steps"]

    prompt = get_equivalence_prompt(qA, stepsA, qB, stepsB)
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    content = resp.choices[0].message.content

    # extract the <response>…</response> block
    m = re.search(r"<response>(.*?)</response>", content, re.DOTALL)
    if not m: return False

    body = m.group(1)

    # grab question flag
    q_m = re.search(r"<question>\s*([YN])\s*</question>", body)
    question_flag = q_m.group(1) if q_m else "N"

    # grab all step flags into a list
    step_flags = [v for _, v in re.findall(r"<step\d+>\s*([YN])\s*</step\d+>", body)]

    # final check: question + every step must be "Y"
    all_flags = [question_flag] + step_flags
    return all(f == "Y" for f in all_flags)

def prm_scorer(questions, steps, client, model, batch_size=32):
    num_samples = len(df)
    tokenizer = AutoTokenizer.from_pretrained(model)

    input_ids_all = []
    token_mask_all = []
    all_rewards = []

    for data in tqdm(df.T.to_dict().values(), total=len(df), desc="[PRM] Preparing input"):
        input_ids, token_mask = prepare_input(
                                model, 
                                problem=questions, 
                                steps=steps, 
                                tokenizer=tokenizer,
                                convert_to_list=True
        )
        input_ids_all.append(input_ids)
        token_mask_all.append(token_mask)


    for start_idx in tqdm(range(0, num_samples, batch_size), desc="[PRM] Scoring"):
        end_idx = start_idx + batch_size
    
        batch_input_ids = input_ids_all[start_idx:end_idx]
        batch_token_masks = token_mask_all[start_idx:end_idx]
    
        batch_input_ids, batch_token_masks = prepare_batch_input_for_model(batch_input_ids, batch_token_masks, pad_token_id=0)
    
        batch_logits = client.embeddings.create(
            input=batch_input_ids.cpu().tolist(),
            model=model,
        )
    
        rewards = derive_step_rewards_vllm(
            model,
            batch_logits,
            batch_token_masks,
            tokenizer
        )
    
        all_rewards.extend(rewards)
    return all_rewards

In [ ]:
def attack(df_sample, 
    task_text, 
    prm_client, 
    experiment_name, 
    causal_client=causal_client
):

    causal_model = causal_client.models.list().data[0].id
    prm_model = prm_client.models.list().data[0].id

    partial_augmentor = partial(augmentor, 
                                task_text=task_text, 
                                client=causal_client, 
                                model=causal_model)

    # Apply the augmentor function to each row in the DataFrame
    with Pool(cpu_count()-2) as pool:
        aug_results = list(
            tqdm(
                pool.imap(partial_augmentor, df_sample.iterrows()),
                total=len(df_sample),
                desc="Augmenting",
            )
        )
    df_aug = pd.DataFrame(aug_results)

    # Check equivalence
    pairs = list(zip(df_sample.iterrows(), df_aug.iterrows()))
    partial_equivalence_check = partial(equivalence_check, 
                                        client=causal_client, 
                                        model=causal_model)

    with Pool(cpu_count()-2) as pool:
        equivalence_results = list(
            tqdm(
                pool.imap(partial_equivalence_check, pairs),
                total=len(pairs),
                desc="Checking equivalence",
            )
        )
    df_aug["equivalence"] = equivalence_results

    # PRM Scorer
    rewards = prm_scorer(questions=df_aug["aug_question"].tolist(), 
                        steps=df_aug["aug_steps"].tolist(), 
                        client=prm_client, model=prm_model)
    df_aug[f"{prm_model}--rewards"] = rewards
    
    # concat the original and augmented DataFrames
    df_sample = pd.concat([df_sample, df_aug], axis=1)
    
    # Save the DataFrame to a CSV file
    df_sample.to_parquet(f"experiments/{experiment_name}.parquet", index=False)

In [ ]:
sample_size = 10
df_sample = df.sample(sample_size)

attack(df_sample, 
    task_text=VERBOSE_TASK, 
    prm_client=qwen_prm_client, 
    experiment_name="verbose_task"
)